# CaseNote AI: run the evaluation

This notebook runs CaseNote AI on the 12 synthetic cases and scores it against the expert labels.

**Before you start:** add your API key as a Colab secret. Click the 🔑 key icon in the left sidebar, add a secret named `GEMINI_API_KEY` (or `ANTHROPIC_API_KEY`), paste the key as its value and turn on *Notebook access*. Never paste a key into a code cell.

All data in this project is synthetic.

In [ ]:
# 1. Get the code
!git clone -q https://github.com/tugbasahingunes/casenote-ai.git
%cd casenote-ai
!pip install -q -r requirements.txt

In [ ]:
# 2. Choose the model provider and load the key from Colab secrets
import os
from google.colab import userdata

PROVIDER = "gemini"   # or "anthropic"
KEY_NAME = {"gemini": "GEMINI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}[PROVIDER]

os.environ["CASENOTE_PROVIDER"] = PROVIDER
os.environ[KEY_NAME] = userdata.get(KEY_NAME)
# Optional: pick a specific model, e.g. os.environ["CASENOTE_MODEL"] = "gemini-3.1-flash-lite"
print("Provider:", PROVIDER)

In [ ]:
# 3. Try one note first (optional, takes up to a minute)
!python src/casenote.py "Mrs. H (83) lives alone. Her nephew has her bank codes. When he stepped out she said quietly: 'Please don't tell him I said anything.'"

In [ ]:
# 4. Run all 12 cases
# Progress is saved after each case. If the model is busy, the code waits and retries.
# If it still stops, just run this cell again: it continues where it left off.
!python src/run_cases.py

In [ ]:
# 5. Score against the expert labels
import sys
sys.path.insert(0, "src")
from casenote import PROVIDERS
model = os.environ.get("CASENOTE_MODEL", PROVIDERS[PROVIDER][1])
latest = f"eval/results/predictions_{model}.json"
!python eval/evaluate.py {latest}

In [ ]:
# 6. Download the predictions and report to upload to GitHub
from google.colab import files
files.download(latest)
files.download(f"eval/results/report_{model}.md")